<a href="https://colab.research.google.com/github/maliksonmez/arnavutkoy-air-quality-forecast/blob/main/notebooks/05_cams_correction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 05 — CAMS düzeltme deneyi

**Soru:** CAMS tahminini yerel ölçüm ve meteorolojiyle düzelten bir model, ertesi günün kirletici seviyesini hem "yarın = bugün" kıyasından hem de ham CAMS'ten daha iyi tahmin edebilir mi?

**Kurallar**
- Bölme `02_forecast_baseline_ridge.ipynb` ile aynı: öğrenme → 06.07.2025, doğrulama 07.07–17.12.2025, test 18.12.2025 → (**dokunulmadı**).
- Her kirletici için ayrı Ridge; alpha yalnızca öğrenme döneminde 5 katlı `TimeSeriesSplit` ile seçilir.
- **E1:** yalnızca bugünün sonunda bilinenler (bugünün CAMS'i). **E2:** CAMS'in hedef gün değeri eklenir (CAMS post-processing).
- Sınırlılık: Open-Meteo arşivi her saat için en güncel model çalışmasını tutar; E2'deki "yarın" CAMS değeri gerçekte bugün elde edilebilecek olandan daha kısa öncelikli bir tahmindir → E2 iyimser, E1 temkinli sınırdır.

In [1]:
import os
import numpy as np
import pandas as pd

# Colab'da GitHub'dan, yerelde repodaki dosyadan okur.
DEPO = "https://github.com/maliksonmez/arnavutkoy-air-quality-forecast/raw/main/data/processed/"
YEREL = "../data/processed/"

def oku(ad):
    kaynak = YEREL + ad if os.path.exists(YEREL + ad) else DEPO + ad
    return pd.read_csv(kaynak, parse_dates=["tarih"], index_col="tarih").loc["2023-09-16":"2026-09-16"]

saatlik = oku("arnavutkoy_hourly_merged.csv")
gunluk = oku("arnavutkoy_daily_merged.csv")
kirleticiler = ["PM10", "PM2_5", "SO2", "NO2"]
print("Saatlik:", saatlik.shape, "| Günlük:", gunluk.shape)

import warnings
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error as mae
warnings.filterwarnings("ignore")

OGR_SON, DOG_BAS, DOG_SON = "2025-07-06", "2025-07-07", "2025-12-17"
ALPHALAR = [0.1, 1, 10, 100, 1000]

# Günlük "ESE–S payı": rüzgârın yol/kentsel alan yönünden estiği saatlerin oranı (04'teki bulgu)
yon = saatlik["met_wind_direction_10m"]
gunluk["met_pay_ESE_S"] = ((yon >= 101.25) & (yon < 191.25)).resample("D").mean()

Saatlik: (26328, 18) | Günlük: (1097, 28)


## 1. A1 sonucunu yeniden üret (kontrol)

In [2]:
g = gunluk[[f"{k}_medyan" for k in kirleticiler]]
g.columns = kirleticiler
X28 = pd.concat([g.shift(L).add_suffix(f"_L{L}") for L in range(1, 8)], axis=1).iloc[7:]
y28 = g.iloc[7:]
ayrim = int(len(X28) * 0.75)
dsinir = int(ayrim * 0.80)
Xo, Xv, yo, yv = X28.iloc[:dsinir], X28.iloc[dsinir:ayrim], y28.iloc[:dsinir], y28.iloc[dsinir:ayrim]

a1 = []
for k in kirleticiler:
    mo = Xo.notna().all(axis=1) & yo[k].notna()
    mv = Xv.notna().all(axis=1) & yv[k].notna()
    en_iyi = min(
        mae(yv[k][mv], make_pipeline(StandardScaler(), Ridge(alpha=a)).fit(Xo[mo], yo[k][mo]).predict(Xv[mv]))
        for a in ALPHALAR
    )
    a1.append({"Kirletici": k, "Gün": int(mv.sum()), "Ridge (28 lag)": en_iyi,
               "Kıyas": mae(yv[k][mv], Xv[f"{k}_L1"][mv])})
display(pd.DataFrame(a1).round(2))   # A1 PDF'indeki değerlerle aynı olmalı

,Kirletici,Gün,Ridge (28 lag),Kıyas
0,PM10,49,7.67,7.57
1,PM2_5,49,3.40,2.63
2,SO2,48,1.51,1.19
3,NO2,49,11.42,9.58


## 2. Girdiler

In [3]:
def girdiler(d, k, cams_yarin):
    """Hedef: d+1 günü medyanı. Girdiler d gününün sonunda bilinenler (+ istenirse CAMS'in d+1 değeri)."""
    t = d[f"{k}_medyan"]
    X = pd.DataFrame(index=d.index)
    X["L1"] = t                        # bugün
    X["L2"] = t.shift(1).fillna(t)     # dün; eksikse bugünkü değer (sabit kural)
    X["L7"] = t.shift(6).fillna(t)     # hedefle aynı hafta günü, bir hafta önce
    for c in ["met_temperature_2m", "met_relative_humidity_2m", "met_surface_pressure",
              "met_wind_speed_10m", "met_wind_u", "met_wind_v", "met_precipitation", "met_pay_ESE_S"]:
        X[c] = d[c]
    for c in [f"cams_{k}_ort"] + (["cams_dust_ort"] if k.startswith("PM") else []):
        X[c + "_bugun"] = d[c]
        if cams_yarin:
            X[c + "_yarin"] = d[c].shift(-1)
    hedef_gun = d.index + pd.Timedelta(days=1)
    X["pazar"] = (hedef_gun.dayofweek == 6).astype(float)
    X["cumartesi"] = (hedef_gun.dayofweek == 5).astype(float)
    X["ay_sin"] = np.sin(2 * np.pi * hedef_gun.month / 12)
    X["ay_cos"] = np.cos(2 * np.pi * hedef_gun.month / 12)
    y = t.shift(-1)
    X.index = y.index = hedef_gun      # satırlar hedef güne göre etiketli
    return X, y


def ridge_kur(Xo, yo):
    tss = TimeSeriesSplit(n_splits=5)
    skor = {a: np.mean([mae(yo.iloc[va], make_pipeline(StandardScaler(), Ridge(alpha=a))
                            .fit(Xo.iloc[tr], yo.iloc[tr]).predict(Xo.iloc[va]))
                        for tr, va in tss.split(Xo)]) for a in ALPHALAR}
    a = min(skor, key=skor.get)
    return make_pipeline(StandardScaler(), Ridge(alpha=a)).fit(Xo, yo), a

## 3. Deney

In [4]:
sonuc, tahminler, katsayilar = [], {}, {}
for k in kirleticiler:
    t = {}
    for ad, yarin in [("E1", False), ("E2", True)]:
        X, y = girdiler(gunluk, k, yarin)
        tamam = X.notna().all(axis=1) & y.notna()
        model, alpha = ridge_kur(X[tamam].loc[:OGR_SON], y[tamam].loc[:OGR_SON])
        Xv = X[tamam].loc[DOG_BAS:DOG_SON]
        t[ad] = pd.Series(model.predict(Xv), index=Xv.index)
        if ad == "E2":
            katsayilar[k] = pd.Series(model[-1].coef_, index=X.columns)
    X2, y2 = girdiler(gunluk, k, True)
    c = X2[f"cams_{k}_ort_yarin"]
    tc = c.notna() & y2.notna()
    lin = LinearRegression().fit(c[tc].loc[:OGR_SON].to_frame(), y2[tc].loc[:OGR_SON])
    cv = c.loc[DOG_BAS:DOG_SON]
    t["Ham CAMS"] = cv
    t["CAMS doğrusal düzeltme"] = pd.Series(lin.predict(cv.to_frame()), index=cv.index)
    t["Kıyas"] = X2["L1"].loc[DOG_BAS:DOG_SON]
    hedef = y2.loc[DOG_BAS:DOG_SON]
    ortak = hedef.notna()
    for m in t:
        ortak &= t[m].reindex(hedef.index).notna()
    gunler = hedef.index[ortak]
    satir = {"Kirletici": k, "Gün": len(gunler), "Toplam gün": len(hedef)}
    for m in ["Kıyas", "Ham CAMS", "CAMS doğrusal düzeltme", "E1", "E2"]:
        satir[m] = mae(hedef[gunler], t[m][gunler])
    sonuc.append(satir)
    tahminler[k] = pd.DataFrame({m: t[m].reindex(gunler) for m in t}).assign(gercek=hedef[gunler])

tablo = pd.DataFrame(sonuc).set_index("Kirletici")
display(tablo.round(2))
print("E2'nin kıyasa göre değişimi (%):", (tablo["E2"] / tablo["Kıyas"] * 100 - 100).round(1).to_dict())
print("E2'nin ham CAMS'e göre değişimi (%):", (tablo["E2"] / tablo["Ham CAMS"] * 100 - 100).round(1).to_dict())

,Gün,Toplam gün,Kıyas,Ham CAMS,CAMS doğrusal düzeltme,E1,E2
Kirletici,,,,,,,
PM10,144,164,7.62,8.72,8.14,7.12,5.04
PM2_5,147,164,2.23,3.26,7.00,2.44,1.96
SO2,111,164,1.39,5.20,5.32,1.58,1.65
NO2,129,164,9.89,13.14,7.88,8.68,5.48


E2'nin kıyasa göre değişimi (%): {'PM10': -33.8, 'PM2_5': -11.8, 'SO2': 19.3, 'NO2': -44.6}
E2'nin ham CAMS'e göre değişimi (%): {'PM10': -42.1, 'PM2_5': -39.7, 'SO2': -68.2, 'NO2': -58.3}


## 4. Kazanç aylara yayılmış mı?

In [5]:
for k in ["PM10", "PM2_5", "NO2"]:
    p = tahminler[k]
    hata = p[["Kıyas", "Ham CAMS", "E1", "E2"]].sub(p["gercek"], axis=0).abs()
    print(k)
    display(hata.groupby(hata.index.to_period("M")).mean().round(2).T)

PM10


tarih,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12
Kıyas,5.54,3.14,4.48,8.34,11.08,14.40
Ham CAMS,5.21,5.47,14.01,8.41,6.31,19.83
E1,4.83,5.02,3.81,7.46,10.02,11.61
E2,3.58,3.37,3.27,5.53,7.11,7.45


PM2_5


tarih,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12
Kıyas,1.52,0.93,0.78,2.61,3.62,3.91
Ham CAMS,2.03,1.65,1.08,2.84,6.61,5.14
E1,1.38,1.55,1.90,2.84,3.33,3.93
E2,1.21,1.13,1.17,2.21,3.14,2.91


NO2


tarih,2025-07,2025-08,2025-09,2025-10,2025-11,2025-12
Kıyas,6.45,2.53,7.03,13.64,16.60,16.46
Ham CAMS,13.36,7.86,12.29,12.68,12.29,26.70
E1,5.26,4.22,6.55,10.39,14.87,13.25
E2,4.40,3.07,4.92,6.66,6.41,8.78


## 5. Neden PM2.5'te sadece CAMS'le kurulan düzeltme başarısız?

In [6]:
X2, y2 = girdiler(gunluk, "PM2_5", True)
for ad, (bas, son) in {"öğrenme": ("2023-09-24", OGR_SON), "doğrulama": (DOG_BAS, DOG_SON)}.items():
    o = pd.concat([X2["cams_PM2_5_ort_yarin"], y2], axis=1).loc[bas:son].dropna()
    print(f"{ad}: istasyon ort. {o.iloc[:, 1].mean():.1f} | CAMS ort. {o.iloc[:, 0].mean():.1f}")

öğrenme: istasyon ort. 16.4 | CAMS ort. 11.7
doğrulama: istasyon ort. 8.9 | CAMS ort. 10.8


İstasyonun PM2.5 seviyesi doğrulama döneminde düşmüş, CAMS ise aynı kalmış. Öğrenme döneminden çıkan sabit düzeltme bu kaymayı kaçırıyor; istasyonun son ölçümünü de kullanan E2 uyum sağlıyor.

## 6. E2'de en etkili girdiler (standartlaştırılmış katsayı büyüklüğü)

In [7]:
display(pd.DataFrame({k: v.abs() for k, v in katsayilar.items()}).round(2)
        .sort_values("NO2", ascending=False).head(8))

,PM10,PM2_5,SO2,NO2
cams_NO2_ort_yarin,NaN,NaN,NaN,8.70
L1,10.04,5.46,2.09,6.20
cams_NO2_ort_bugun,NaN,NaN,NaN,2.88
met_temperature_2m,1.88,0.82,0.37,2.29
ay_cos,1.64,0.17,0.28,2.09
met_wind_speed_10m,0.29,0.07,0.07,1.74
L7,1.97,1.13,0.01,1.67
met_surface_pressure,0.13,0.07,0.13,1.66


## Sonuç

- E2 (CAMS + istasyon + meteoroloji) PM10, PM2.5 ve NO2'de hem kıyası hem ham CAMS'i geçiyor; kazanç en çok sonbahar–kışta.
- E1 (yalnızca bugün bilinenler) PM10 ve NO2'de kıyası geçiyor, PM2.5'te geçemiyor.
- SO2'de hiçbir model kıyası geçemiyor; SO2 düşük ve WHO eşiğini hiç aşmıyor → öncelik düşük.
- Değerlendirilebilen gün sayısı 48–49'dan 111–147'ye çıktı (her kirletici kendi geçmişini kullandığı için).
- **Ödev 2:** gerçek gün-öncesi CAMS çalışmalarıyla (ADS) yeniden değerlendirme, Random Forest/XGBoost, kayan pencere doğrulaması; test dönemi yalnızca en sonda bir kez kullanılacak.